# House Price Prediction using Machine Learning

## Overview
This notebook builds a machine learning model to predict house prices based on property features.

## Workflow
1. Data Loading & Exploration
2. Data Cleaning & Preprocessing
3. Exploratory Data Analysis (EDA)
4. Feature Engineering & Selection
5. Model Training (Linear Regression & Random Forest)
6. Model Evaluation & Comparison
7. Results & Insights

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Libraries imported successfully!')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/harsh317/House-Prices---Advanced-Regression-Techniques-KAGGLE/master/train.csv')

print(f'Dataset shape: {df.shape}')
print(f'\\nFirst few rows:')
print(df.head())
print(f'\\nDataset info:')
print(df.info())

## 3. Data Cleaning

In [ ]:
df_clean = df.copy()

numerical_cols = df_clean.select_dtypes(include=['int64', 'float64']).columns
for col in numerical_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

categorical_cols = df_clean.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)

df_encoded = pd.get_dummies(df_clean, drop_first=True)
print(f'Shape after preprocessing: {df_encoded.shape}')

## 4. Feature Engineering

In [ ]:
X = df_encoded.drop(['SalePrice', 'Id'], axis=1)
y = df_encoded['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training set size: {X_train.shape[0]}')
print(f'Test set size: {X_test.shape[0]}')

## 5. Linear Regression Model

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)

lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
lr_mae = mean_absolute_error(y_test, y_pred_lr)
lr_r2 = r2_score(y_test, y_pred_lr)

print('Linear Regression Results:')
print(f'Test RMSE: {lr_rmse:,.2f}')
print(f'Test MAE: {lr_mae:,.2f}')
print(f'Test R² Score: {lr_r2:.4f}')

## 6. Random Forest Model

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_r2 = r2_score(y_test, y_pred_rf)

print('Random Forest Results:')
print(f'Test RMSE: {rf_rmse:,.2f}')
print(f'Test MAE: {rf_mae:,.2f}')
print(f'Test R² Score: {rf_r2:.4f}')

## 7. Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'Test RMSE': [lr_rmse, rf_rmse],
    'Test MAE': [lr_mae, rf_mae],
    'Test R² Score': [lr_r2, rf_r2]
})

print('\\nModel Comparison:')
print(comparison)

best = 'Random Forest' if rf_rmse < lr_rmse else 'Linear Regression'
print(f'\\nBest Model: {best}')

## 8. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred_lr, alpha=0.6, label='Linear Regression')
axes[0].scatter(y_test, y_pred_rf, alpha=0.6, label='Random Forest')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Price')
axes[0].set_ylabel('Predicted Price')
axes[0].set_title('Actual vs Predicted Prices')
axes[0].legend()
axes[0].grid(alpha=0.3)

models = ['Linear\\nRegression', 'Random\\nForest']
rmse_scores = [lr_rmse, rf_rmse]
axes[1].bar(models, rmse_scores, color=['#3498db', '#2ecc71'])
axes[1].set_ylabel('Test RMSE')
axes[1].set_title('Model Performance Comparison')

plt.tight_layout()
plt.show()

print('Visualizations generated successfully!')

## 9. Save Models

In [ ]:
import pickle
import os

os.makedirs('models', exist_ok=True)

with open('models/linear_regression.pkl', 'wb') as f:
    pickle.dump(lr_model, f)

with open('models/random_forest.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('✓ Models saved successfully!')

## 10. Summary

In [ ]:
print('='*60)
print('HOUSE PRICE PREDICTION - FINAL RESULTS')
print('='*60)
print(f'\\nDataset: {len(df)} houses')
print(f'Features: {X.shape[1]}')
print(f'Price Range: ${y.min():,.0f} - ${y.max():,.0f}')
print(f'Average Price: ${y.mean():,.0f}')
print('\\n' + '='*60)
print('BEST MODEL: Random Forest' if rf_rmse < lr_rmse else 'BEST MODEL: Linear Regression')
print('='*60)